In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Verification Gate
input_file = "master_feature_table_with_hazards.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Missing framework feature bank! Please run Notebook 06 first.")

df_master = pd.read_csv(input_file)
print("=====================================================================")
print("📊 DASHBOARD DATA LAYER: FRAMEWORK MATRICES INGESTED")
print("=====================================================================")
print(f"Verified Records Footprint: {df_master.shape[0]} rows (Expected: 18260)")

📊 DASHBOARD DATA LAYER: FRAMEWORK MATRICES INGESTED
Verified Records Footprint: 18260 rows (Expected: 18260)


## Data Compaction & Multi-Tier KPI Profile Exports

In [3]:
# 1. Select and export the clean tourist-friendly timeline dataset
dashboard_columns = [
    'location', 'date', 'overall_hazard_score', 'risk_category',
    'weather_risk_share', 'landslide_risk_share', 'crowd_risk_share', 'transport_risk_share'
]
df_dashboard_out = df_master[dashboard_columns].copy()

# Round continuous floating-point fractions for clean UI rendering
for col in ['overall_hazard_score', 'weather_risk_share', 'landslide_risk_share', 'crowd_risk_share', 'transport_risk_share']:
    df_dashboard_out[col] = df_dashboard_out[col].round(4)

df_dashboard_out.to_csv("risk_attribution_dashboard.csv", index=False)
print("🎉 Success! Pristine tracking matrix exported to 'risk_attribution_dashboard.csv'")

🎉 Success! Pristine tracking matrix exported to 'risk_attribution_dashboard.csv'


In [4]:
# 2. Compute and save Global Macro Shares Reference Table
global_shares = {
    "Weather": df_master["weather_risk_share"].mean(),
    "Landslide": df_master["landslide_risk_share"].mean(),
    "Crowd": df_master["crowd_risk_share"].mean(),
    "Transport": df_master["transport_risk_share"].mean()
}
df_global_shares = pd.DataFrame({
    "Channel": global_shares.keys(),
    "Global Average Share (%)": [round(val * 100.0, 2) for val in global_shares.values()]
}).sort_values(by="Global Average Share (%)", ascending=False).reset_index(drop=True)

df_global_shares.to_csv("global_dashboard_risk_shares.csv", index=False)
print("🎉 Success! Global macro-share profiles written to 'global_dashboard_risk_shares.csv'")

🎉 Success! Global macro-share profiles written to 'global_dashboard_risk_shares.csv'


In [5]:
# 3. Build City-Level Risk Share Attribution Profiles Matrix
share_cols = ["weather_risk_share", "landslide_risk_share", "crowd_risk_share", "transport_risk_share"]
df_city_shares = (df_master.groupby("location")[share_cols].mean() * 100.0).round(2).reset_index()
df_city_shares.columns = ["Location", "Weather Risk Share (%)", "Landslide Risk Share (%)", "Crowd Risk Share (%)", "Transport Risk Share (%)"]

df_city_shares.to_csv("city_risk_share_profiles.csv", index=False)
print("🎉 Success! Regional geographic signatures written to 'city_risk_share_profiles.csv'")

🎉 Success! Regional geographic signatures written to 'city_risk_share_profiles.csv'


## Dominant Risk Share City Ranking Generator

In [6]:
channels_to_rank = {
    "Weather": "Weather Risk Share (%)",
    "Landslide": "Landslide Risk Share (%)",
    "Crowd": "Crowd Risk Share (%)",
    "Transport": "Transport Risk Share (%)"
}

ranking_records = []

In [7]:
# Loop across each channel to compile a clean sorted leaderboard stack
for channel_name, col_identifier in channels_to_rank.items():
    # Sort locations descending by their vulnerability share density
    df_sorted = df_city_shares.sort_values(by=col_identifier, ascending=False).reset_index(drop=True)
    
    # Using iterrows() completely eliminates any naming or special character attribute bugs!
    for rank_idx, (idx, row) in enumerate(df_sorted.iterrows(), 1):
        ranking_records.append({
            "Risk Category Focus": channel_name,
            "Rank Position": rank_idx,
            "Location": row["Location"],
            "Channel Contribution Share (%)": row[col_identifier]
        })

In [8]:
df_rankings_out = pd.DataFrame(ranking_records)
output_rankings_csv = "city_vulnerability_rankings.csv"
df_rankings_out.to_csv(output_rankings_csv, index=False)

print(f"🎉 Success! Multi-channel city vulnerability lists exported to '{output_rankings_csv}'")

🎉 Success! Multi-channel city vulnerability lists exported to 'city_vulnerability_rankings.csv'


## Quality Assurance Validation Audit

In [9]:
print("=====================================================================")
print("🏁 DASHBOARD LAYER DATA AUDIT VALIDATION QUALITY REPORT")
print("=====================================================================")

# Assert that city-level additivity sums match 100% bounds
for idx, row in df_city_shares.iterrows():
    row_sum = row["Weather Risk Share (%)"] + row["Landslide Risk Share (%)"] + row["Crowd Risk Share (%)"] + row["Transport Risk Share (%)"]
    assert np.isclose(row_sum, 100.0, atol=1e-1), f"QA Failure: Row total for {row['Location']} sums to {row_sum}% instead of 100%"

print(" ✅ All programmatic city-level share additivity constraints passed.")
print(" ✅ Programmatic ranking grid successfully compiled and locked.")

print("\n📊 DASHBOARD CARD PREVIEW: TOP 3 MOST SENSITIVE CITIES BY CHANNEL")
print("---------------------------------------------------------------------")

for channel in ["Weather", "Landslide", "Crowd", "Transport"]:
    print(f"\n🔥 Top Exposed Destinations for [{channel} Hazard Channel]:")
    sub_set = df_rankings_out[(df_rankings_out["Risk Category Focus"] == channel) & (df_rankings_out["Rank Position"] <= 3)]
    for idx, row in sub_set.iterrows():
        print(f"   Rank {row['Rank Position']}: {row['Location']:<12} ({row['Channel Contribution Share (%)']:.2f}%)")

print("\n=====================================================================")
print("🚀 NOTEBOOK 07C PIPELINE CLOSED: FULLY PREPARED FOR THE UI CONTEXT")
print("=====================================================================")

🏁 DASHBOARD LAYER DATA AUDIT VALIDATION QUALITY REPORT
 ✅ All programmatic city-level share additivity constraints passed.
 ✅ Programmatic ranking grid successfully compiled and locked.

📊 DASHBOARD CARD PREVIEW: TOP 3 MOST SENSITIVE CITIES BY CHANNEL
---------------------------------------------------------------------

🔥 Top Exposed Destinations for [Weather Hazard Channel]:
   Rank 1: Goa          (18.43%)
   Rank 2: Jaipur       (15.62%)
   Rank 3: Munnar       (10.57%)

🔥 Top Exposed Destinations for [Landslide Hazard Channel]:
   Rank 1: Munnar       (26.61%)
   Rank 2: Darjeeling   (22.17%)
   Rank 3: Goa          (20.38%)

🔥 Top Exposed Destinations for [Crowd Hazard Channel]:
   Rank 1: Jaipur       (44.92%)
   Rank 2: Goa          (35.50%)
   Rank 3: Mussoorie    (22.94%)

🔥 Top Exposed Destinations for [Transport Hazard Channel]:
   Rank 1: Leh          (58.13%)
   Rank 2: Mussoorie    (52.72%)
   Rank 3: Shimla       (52.17%)

🚀 NOTEBOOK 07C PIPELINE CLOSED: FULLY PREPARED 